# Instalar dependencias

In [2]:
%pip install -U langchain-community

%pip install pypdf


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Lectura PDF

In [3]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Chunks

In [4]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Transformar texto y vectores

In [5]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

def vector_to_text(vector):
  # Cargar modelo
  model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

  # Convertir vector en texto
  text = model.decode(vector)

  return text

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chunks en vectores

In [6]:
chunk_vectors = []
chunk_vectors_len = []
for chunk in chunks:
  vector = text_to_vector(chunk)
  chunk_vectors.append(vector)
  chunk_vectors_len.append(len(vector))

print(len(chunk_vectors))
print(chunk_vectors_len)

264
[384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 

# Guardar Chunks

In [7]:
import numpy as np

# Convertir a un array de NumPy y guardar
np.save("chunks.npy", np.array(chunk_vectors))

# Leer desde el archivo .npy
loaded_vectors = np.load("chunks.npy")

print("Vectores cargados:", loaded_vectors)


Vectores cargados: [[ 0.01517293  0.07472266  0.03041344 ...  0.10975496 -0.02085184
  -0.01560559]
 [ 0.01927043  0.05052567  0.04363388 ...  0.1349137  -0.01790163
  -0.07349641]
 [ 0.01779155 -0.01842804 -0.01193615 ...  0.1139394  -0.00146952
  -0.11986542]
 ...
 [ 0.02725722  0.06018103 -0.0123213  ...  0.06511977 -0.03400579
  -0.05573653]
 [-0.02108937  0.10245892  0.00366546 ...  0.10552087 -0.01468905
  -0.06402979]
 [-0.00938701  0.08433828  0.02082387 ...  0.04647428 -0.04696859
  -0.07766555]]


# Instalar SWIG

In [10]:
!apt-get install swig

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: command not found: apt-get


# Ball Tree

## Headers

In [17]:
%%file ball_tree.h

#ifndef BALL_TREE_H
#define BALL_TREE_H

#include <vector>

class BallTree {
public:
    BallTree();
    void build(const std::vector<std::vector<double> >& points);
    std::vector<double> find_nearest(const std::vector<double>& target);
};

#endif // BALL_TREE_H


Overwriting ball_tree.h


## Class

In [18]:
%%file ball_tree.cpp
#include "ball_tree.h"
#include <cmath>
#include <limits>
#include <algorithm>

struct Node {
    std::vector<double> point;
    Node* left;
    Node* right;
};

Node* root;
std::vector<std::vector<double> > data;

double euclidean_distance(const std::vector<double>& a, const std::vector<double>& b) {
    double sum = 0.0;
    for (size_t i = 0; i < a.size(); ++i) {
        sum += (a[i] - b[i]) * (a[i] - b[i]);
    }
    return std::sqrt(sum);
}

Node* build_tree(int left, int right, int depth) {
    if (left > right) return nullptr;

    int mid = (left + right) / 2;
    std::nth_element(data.begin() + left, data.begin() + mid, data.begin() + right + 1,
                     [depth](const std::vector<double>& a, const std::vector<double>& b) {
                         return a[depth % a.size()] < b[depth % a.size()];
                     });

    Node* node = new Node{data[mid], nullptr, nullptr};
    node->left = build_tree(left, mid - 1, depth + 1);
    node->right = build_tree(mid + 1, right, depth + 1);
    return node;
}

void nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth) {
    if (!node) return;

    double dist = euclidean_distance(target, node->point);
    if (dist < best_dist) {
        best_dist = dist;
        best = node;
    }

    int axis = depth % target.size();
    Node* next = target[axis] < node->point[axis] ? node->left : node->right;
    Node* other = next == node->left ? node->right : node->left;

    nearest_neighbor(next, target, best, best_dist, depth + 1);

    if (std::abs(target[axis] - node->point[axis]) < best_dist) {
        nearest_neighbor(other, target, best, best_dist, depth + 1);
    }
}

BallTree::BallTree() : root(nullptr) {}

void BallTree::build(const std::vector<std::vector<double>>& points) {
    data = points;
    root = build_tree(0, data.size() - 1, 0);
}

std::vector<double> BallTree::find_nearest(const std::vector<double>& target) {
    Node* best = nullptr;
    double best_dist = std::numeric_limits<double>::max();
    nearest_neighbor(root, target, best, best_dist, 0);
    return best ? best->point : std::vector<double>();
}


Overwriting ball_tree.cpp


## Interfaz

In [19]:
%%file ball_tree.i
%module ball_tree
%{
#include "ball_tree.h"
%}

%include "std_vector.i"
%template(VectorDouble) std::vector<double>;
%template(VectorVectorDouble) std::vector<std::vector<double>>;

%include "ball_tree.h"


Overwriting ball_tree.i


# Ejecutar SWIG

In [28]:
!swig -c++ -python ball_tree.i
!g++ -O2 -fPIC -c ball_tree.cpp
!g++ -O2 -fPIC -c ball_tree_wrap.cxx -I/usr/include/python3.10
!g++ -shared ball_tree.o ball_tree_wrap.o -o _ball_tree.so


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ball_tree.cpp:28:22: error: expected expression
   28 |                      [depth](const std::vector<double>& a, const std::vector<double>& b) {
      |                      ^
ball_tree.cpp:32:26: error: expected ';' at end of declaration
   32 |     Node* node = new Node{data[mid], nullptr, nullptr};
      |                          ^
      |                          ;
ball_tree.cpp:58:24: error: member initializer 'root' does not name a non-static data member or base class
   58 | BallTree::BallTree() : root(nullptr) {}
      |                        ^~~~~~~~~~~~~
ball_tree.cpp:60:58: error: a space is required between consecutive right angle brackets (use '> >')
   60 | void BallTree::build(const std::vector<std::vector<double>>& points) {
      |                                                          ^~
      |                                                          > >
4 errors generated.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ball_tree_wrap.cxx:203:11: fatal error: 'Python.h' file not found
  203 | # include <Python.h>
      |           ^~~~~~~~~~
1 error generated.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


clang++: error: no such file or directory: 'ball_tree.o'
clang++: error: no such file or directory: 'ball_tree_wrap.o'
clang++: error: no input files
